In [6]:
# Import necessary libraries
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv1D, Flatten, Dense, Dropout, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import kneighbors_graph
from spektral.layers import GCNConv
import matplotlib.pyplot as plt

# ---------------------------
# Step 1: Data Collection & Preprocessing (GLORIA dataset example)
# ---------------------------

# Load GLORIA Dataset (replace with actual file paths)
meta_data_path = "C:/Users/BraedynM/Documents/Science Fair 2025/GLORIA_dataset/GLORIA_meta_and_lab.csv"
rrs_data_path = "C:/Users/BraedynM/Documents/Science Fair 2025/GLORIA_dataset/GLORIA_Rrs.csv"

# Load metadata and reflectance data
meta_data = pd.read_csv(meta_data_path)
rrs_data = pd.read_csv(rrs_data_path)

# Select relevant spectral bands (e.g., Rrs_443, Rrs_560, Rrs_665)
selected_bands = ['Rrs_443', 'Rrs_560', 'Rrs_665']
X_data = rrs_data[selected_bands].values

# Normalize the features to [0, 1]
scaler = MinMaxScaler()
X_norm = scaler.fit_transform(X_data)

# Extract chlorophyll-a values from metadata (assume the column name is 'Chla')
y_data = meta_data['Chla'].values  # Chlorophyll-a in µg/L

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_norm, y_data, test_size=0.2, random_state=42)

# Reshape X_train for the CNN branch: model expects (n_nodes, n_features, 1)
X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
# --------------------------------------------
# Normalize features
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)

# Create adjacency matrix
from sklearn.neighbors import kneighbors_graph


# Create adjacency matrix
A = kneighbors_graph(X_train, n_neighbors=5, mode='connectivity', include_self=True)
A_dense = A.toarray().astype('float32')







# ---------------------------
# Step 2: Graph Construction
# ---------------------------
# Create an adjacency matrix using k-nearest neighbors (k=5)
A = kneighbors_graph(X_train, n_neighbors=5, mode='connectivity', include_self=True)
A = A.toarray().astype('float32')  # Adjacency matrix shape: (n_train, n_train)
A_tensor = tf.convert_to_tensor(A)  # Convert adjacency matrix to TensorFlow tensor

# ---------------------------
# Step 3: Build the Hybrid GNN-CNN Model
# ---------------------------

# Define two inputs: one for CNN branch and one for GNN branch
input_cnn = Input(shape=(X_train_cnn.shape[1], 1), name='cnn_input')  # CNN input: (features, 1)
input_gnn = Input(shape=(X_train.shape[1],), name='gnn_input')         # GNN input: (features)

# CNN Branch: Apply 1D convolution to extract local feature patterns from each node's feature vector.
cnn_branch = Conv1D(filters=16, kernel_size=2, activation='relu')(input_cnn)
cnn_branch = Flatten()(cnn_branch)

# GNN Branch: Use a Graph Convolutional Layer to capture spatial relationships across training samples.
gnn_branch = GCNConv(16, activation='relu')([input_gnn, A_tensor])
gnn_branch = Flatten()(gnn_branch)

# Merge the two branches
merged = Concatenate()([cnn_branch, gnn_branch])
dense1 = Dense(64, activation='relu')(merged)
drop1 = Dropout(0.3)(dense1)
output = Dense(1, name='prediction')(drop1)  # Regression output: predicting chlorophyll-a

# Define the model with two inputs
model = Model(inputs=[input_cnn, input_gnn], outputs=output)

# Compile the model with Adam optimizer and MSE loss function
model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
model.summary()

# ---------------------------
# Step 4: Train the Model
# ---------------------------
history = model.fit([X_train_cnn, X_train], y_train,
                    epochs=50,
                    batch_size=16,
                    validation_split=0.2)

# ---------------------------
# Step 5: Model Evaluation and Visualization
# ---------------------------

loss, mae = model.evaluate([X_test_cnn, X_test], y_test)
print(f"\nTest Mean Absolute Error (MAE): {mae}")

# Plot training history: Loss and MAE over epochs
plt.figure(figsize=(12, 6))

# Plot Loss (MSE) over epochs
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.legend()
plt.title('Training vs Validation Loss')

# Plot MAE over epochs
plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='Training MAE')
plt.plot(history.history['val_mae'], label='Validation MAE')
plt.xlabel('Epochs')
plt.ylabel('Mean Absolute Error')
plt.legend()
plt.title('Training vs Validation MAE')

plt.tight_layout()
plt.show()

# ---------------------------
# Step 6: Testing on New Data (Example)
# ---------------------------

new_sample_cnn = X_test_cnn[0].reshape(1, X_test_cnn.shape[1], 1)  # First test sample for CNN branch
new_sample_gnn = X_test[0].reshape(1, X_test.shape[1])             # First test sample for GNN branch

predicted_value = model.predict([new_sample_cnn, new_sample_gnn])
print(f"\nPredicted Chlorophyll-a for new sample: {predicted_value[0][0]:.3f} µg/L")


ValueError: Exception encountered when calling GCNConv.call().

[1mCould not automatically infer the output shape / dtype of 'gcn_conv_2' (of type GCNConv). Either the `GCNConv.call()` method is incorrect, or you need to implement the `GCNConv.compute_output_spec() / compute_output_shape()` method. Error encountered:

Tried to convert 'y' to a tensor and failed. Error: None values not supported.[0m

Arguments received by GCNConv.call():
  • args=(['<KerasTensor shape=(None, 3), dtype=float32, sparse=False, name=gnn_input>', 'tf.Tensor(shape=(6057, 6057), dtype=float32)'],)
  • kwargs={'mask': ['None', 'None']}

In [5]:
import tensorflow as tf
import numpy as np
from spektral.layers import GCNConv

# Example input data for debugging
input_gnn = tf.keras.Input(shape=(3,), name='gnn_input')  # Example input shape (3 features per node)
A_tensor = tf.convert_to_tensor(np.eye(6057), dtype=tf.float32)  # Example adjacency matrix (identity matrix)

# Define the GCNConv layer
gcn_layer = GCNConv(16, activation='relu')

# Try passing the inputs to the GCNConv layer and check for errors
try:
    gnn_branch = gcn_layer([input_gnn, A_tensor])
    print("GCNConv output shape:", gnn_branch.shape)
except Exception as e:
    print("Error in GCNConv layer:", e)


Error in GCNConv layer: Exception encountered when calling GCNConv.call().

Could not automatically infer the output shape / dtype of 'gcn_conv_1' (of type GCNConv). Either the `GCNConv.call()` method is incorrect, or you need to implement the `GCNConv.compute_output_spec() / compute_output_shape()` method. Error encountered:

Tried to convert 'y' to a tensor and failed. Error: None values not supported.

Arguments received by GCNConv.call():
  • args=(['<KerasTensor shape=(None, 3), dtype=float32, sparse=False, name=gnn_input>', 'tf.Tensor(shape=(6057, 6057), dtype=float32)'],)
  • kwargs={'mask': ['None', 'None']}


In [ ]:
import pandas as pd

# File paths for GLORIA dataset
meta_data_path = "C:/Users/BraedynM/Documents/Science Fair 2025/GLORIA_dataset/GLORIA_meta_and_lab.csv"
rrs_data_path = "C:/Users/BraedynM/Documents/Science Fair 2025/GLORIA_dataset/GLORIA_Rrs.csv"

# Load metadata (water quality parameters)
try:
    meta_data = pd.read_csv(meta_data_path)
    print("Metadata sample:")
    print(meta_data.head())
except FileNotFoundError:
    print(f"Error: File not found at {meta_data_path}. Please verify the file path.")

# Load remote sensing reflectance data (Rrs)
try:
    rrs_data = pd.read_csv(rrs_data_path)
    print("\nRrs data sample:")
    print(rrs_data.head())
except FileNotFoundError:
    print(f"Error: File not found at {rrs_data_path}. Please verify the file path.")

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Select relevant spectral bands (e.g., Rrs_443, Rrs_560, Rrs_665)
selected_bands = ['Rrs_443', 'Rrs_560', 'Rrs_665']
X_data = rrs_data[selected_bands].values

# Normalize the features to [0, 1]
scaler = MinMaxScaler()
X_norm = scaler.fit_transform(X_data)

# Extract chlorophyll-a values from metadata (assume column name is 'Chla')
y_data = meta_data['Chla'].values  # Chlorophyll-a in µg/L

print("\nNormalized spectral data sample:")
print(X_norm[:5])
print("\nChlorophyll-a labels sample:")
print(y_data[:5])

In [ ]:
from sklearn.model_selection import train_test_split

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_norm, y_data, test_size=0.2, random_state=42)

print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")


In [ ]:
from sklearn.neighbors import kneighbors_graph
import tensorflow as tf

# Create adjacency matrix using k-nearest neighbors (k=5)
A = kneighbors_graph(X_train, n_neighbors=5, mode='connectivity', include_self=True)
A_dense = A.toarray().astype('float32')  # Convert to dense numpy array

# Convert adjacency matrix to TensorFlow tensor
A_tensor = tf.convert_to_tensor(A_dense)

print("\nAdjacency matrix shape:", A_tensor.shape)


In [ ]:
from tensorflow.keras.layers import Input, Conv1D, Flatten, Dense, Dropout, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from spektral.layers import GCNConv

# Define CNN input and branch
input_cnn = Input(shape=(X_train.shape[1], 1), name='cnn_input')
cnn_branch = Conv1D(filters=16, kernel_size=2, activation='relu')(input_cnn)
cnn_branch = Flatten()(cnn_branch)

# Define GNN input and branch
input_gnn = Input(shape=(X_train.shape[1],), name='gnn_input')
gnn_branch = GCNConv(16, activation='relu')([input_gnn, A_tensor])
gnn_branch = Flatten()(gnn_branch)

# Merge CNN and GNN branches
merged = Concatenate()([cnn_branch, gnn_branch])
dense1 = Dense(64, activation='relu')(merged)
dropout1 = Dropout(0.3)(dense1)
output = Dense(1)(dropout1)  # Regression output

# Define model with two inputs (CNN and GNN branches)
model = Model(inputs=[input_cnn, input_gnn], outputs=output)

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model.summary()


In [ ]:
# Reshape X_train for CNN input (add channel dimension for Conv1D)
X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print("\nReshaped X_train_cnn shape:", X_train_cnn.shape)


In [ ]:
history = model.fit([X_train_cnn, X_train], y_train,
                    epochs=50,
                    batch_size=16,
                    validation_split=0.2)

print("\nTraining complete!")


In [ ]:
loss, mae = model.evaluate([X_test_cnn, X_test], y_test)
print(f"\nTest Loss: {loss}, Test MAE: {mae}")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

# Plot Loss (MSE) over epochs
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.legend()
plt.title('Training vs Validation Loss')

# Plot MAE over epochs
plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='Training MAE')
plt.plot(history.history['val_mae'], label='Validation MAE')
plt.xlabel('Epochs')
plt.ylabel('Mean Absolute Error')
plt.legend()
plt.title('Training vs Validation MAE')

plt.tight_layout()
plt.show()
